Importacion de librerias para manejo del sistema

In [31]:
import sys
import os
from pathlib import Path

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
#Manipulación y Análisis de Datos
import numpy as np
import pandas as pd
import pyreadstat
import plotly.express as px

#Funciones propias estadisticas
from src.utils_pisa import *


In [55]:
ruta_datos_intermedios = Path('../../data/intermediate')
ruta_medias_matematicas = ruta_datos_intermedios/'medias_matematicas_anual'

In [34]:
establecer_semilla()

✅ Semillas del proyecto establecidas con éxito (seed=42).


# Carga de datos y exploracion

Para la lectura de datos, en vez de el metodo de pandas *read_sas* emplearemos [*pyreadstat*](https://github.com/Roche/pyreadstat/tree/master#reading-files) que presenta una mejor API para el manjo de datos SAS obteniendo el dataframe y sus metadatos

Realizamos la lectura de los metadatos, para obtener el nombre e informacion de las columnas 

In [46]:
#Realizamos la lectura de los datos de interes
df, meta = pyreadstat.read_sas7bdat('../../data/raw/CY08MSP_STU_QQQ.SAS7BDAT', metadataonly=True)

In [47]:
print(f'Numero de filas: {meta.number_rows} \nNumero de columnas: {meta.number_columns}')

Numero de filas: 613744 
Numero de columnas: 1278


# Media Matemáticas PISA 2022

In [49]:
columnas_lectura = [col for col in meta.column_names if 'PV' in col and 'MATH' in col]

In [50]:
columnas_id=['CNT','STRATUM','CNTSCHID','CNTSTUID']

In [51]:
print(f"Columnas encontradas: {len(columnas_lectura)}")
print(columnas_lectura)

Columnas encontradas: 10
['PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH', 'PV6MATH', 'PV7MATH', 'PV8MATH', 'PV9MATH', 'PV10MATH']


In [ ]:
#Añadimos el resto de columnas de interes para realizar el estudio : 
columnas_interes =  columnas_id+columnas_lectura+['W_FSTUWT']

df_pisa, meta_real = pyreadstat.read_sas7bdat(
    '../../data/raw/CY08MSP_STU_QQQ.SAS7BDAT', 
    usecols=columnas_interes
)
df_pisa.info(memory_usage='deep')

## Cálculo a nivel de alumno

<div style="
  background: linear-gradient(135deg, rgba(128, 128, 128, 0.25) 0%, rgba(64, 64, 64, 0.15) 100%);
  backdrop-filter: blur(4px);
  -webkit-backdrop-filter: blur(4px);
  border-left: 5px solid rgba(128, 128, 128, 0.6);
  padding: 16px;
  border-radius: 8px;
  color: inherit;
  margin: 20px 0;
">
  <strong>⚠️ ADVERTENCIA METODOLÓGICA: Simplificación de Valores Plausibles (PVs)</strong><br><br>
  Para poder entrenar algoritmos de <i>Machine Learning</i>, que requieren una única variable objetivo por observación, se ha calculado la media aritmética de los 10 Valores Plausibles (PVs) para obtener una sola calificación por alumno.<br><br>
  Esta decisión técnica difiere de las directrices analíticas de la OCDE. Al consolidar la nota a nivel individual, <b>se asume la pérdida de la varianza de imputación</b> y de la incertidumbre de la medición original, lo que resultará en estimaciones de error estándar más ajustadas en los modelos predictivos posteriores.
</div>

In [52]:
df_medias_mat_alumno = puntuaciones_pisa_alumnos(df_pisa,'MATH')

In [53]:
df_medias_mat_alumno.head()

,CNT,CNTSCHID,CNTSTUID,media_math_pisa
0,ALB,800282.0,800001.0,223.0351
1,ALB,800115.0,800002.0,308.4938
2,ALB,800242.0,800003.0,313.7353
3,ALB,800245.0,800005.0,298.7323
4,ALB,800285.0,800006.0,475.7512


In [56]:
df_medias_mat_alumno.to_parquet(ruta_medias_matematicas/'pisa22_medias_math_alumno.parquet')

## Calculo a nivel estrato

In [37]:
df_medias_mat_estrato = puntuaciones_pisa_estratos(df_pisa,'MATH',col_peso='W_FSTUWT')

In [39]:
df_medias_mat_estrato.head()

,CNT,STRATUM,media_math_estrato
0,ALB,ALB01,343.634841
1,ALB,ALB02,414.736267
2,ALB,ALB03,372.771643
3,ALB,ALB04,423.938765
4,ALB,ALB05,368.182410


In [41]:
#Guardamos el dataframe en formato parquet
df_medias_mat_estrato.to_parquet(ruta_datos_intermedios/'pisa22_medias_math_estrato.parquet')

## Cálculo a nivel de pais

In [57]:
rendimiento_matematicas_pais_PISA= puntuaciones_pisa_pais(df_pisa,competencia='MATH')

In [58]:
#Este es el dataframe que contiene cada media de los PV de lectura independiente
rendimiento_matematicas_pais_PISA.head()

,CNT,media_math_pisa,varianza_imputacion
0,ALB,368.221720,0.369265
1,ARE,431.110477,0.127318
2,ARG,377.529029,0.174689
3,AUS,487.084254,0.120797
4,AUT,487.267499,0.171985


In [59]:
rendimiento_matematicas_pais_PISA.to_parquet(ruta_medias_matematicas/'pisa22_medias_math_pais.parquet', index=False)

# Media Matemáticas PISA 2018

In [42]:
#TODO medias por estrato de 2018

# Media Matemáticas PISA 2015

In [43]:
#TODO medias por estrato de 2015

# Media Matemáticas PISA 2012

In [44]:
#TODO medias matematixas 2012